In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# on importes les données

In [2]:
df = pd.read_csv("Telco-Customer-Churn.csv")

### imprimer les info du jeux des donnéess

In [3]:
print("Info du Dataset:\n")
print(df.info())
print("\n Distribution de class: \n")
print(df['Churn'].value_counts())
print("\n Sample Data:\n", df.head())

Info du Dataset:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 n

## on geres les valeurs manquantes

In [4]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.fillna({'TotalCharges': df['TotalCharges'].median()}, inplace=True)

In [7]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# Encoder les variables categoriels

In [8]:
label_encoder = LabelEncoder()
for column in df.select_dtypes(include=['object']).columns:
    if column != 'Churn':
        df[column] = label_encoder.fit_transform(df[column])

In [9]:
# Encoder la variable cible
df['Churn'] = label_encoder.fit_transform(df['Churn'])

## On standarises les valeurs 

In [10]:
scaler = StandardScaler()
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
df[numerical_features] = scaler.fit_transform(df[numerical_features])

## Affecter les variables a X e Y

In [18]:
X = df.drop(columns=['Churn'])
y = df['Churn']

In [21]:
print(X)

      customerID  gender  SeniorCitizen  Partner  Dependents    tenure  \
0           5375       0              0        1           0 -1.277445   
1           3962       1              0        0           0  0.066327   
2           2564       1              0        0           0 -1.236724   
3           5535       1              0        0           0  0.514251   
4           6511       0              0        0           0 -1.236724   
...          ...     ...            ...      ...         ...       ...   
7038        4853       1              0        1           1 -0.340876   
7039        1525       0              0        1           1  1.613701   
7040        3367       0              0        1           1 -0.870241   
7041        5934       1              1        1           0 -1.155283   
7042        2226       1              0        0           0  1.369379   

      PhoneService  MultipleLines  InternetService  OnlineSecurity  \
0                0              1        

In [22]:
print(y)

0       0
1       0
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    1
7042    0
Name: Churn, Length: 7043, dtype: int32


# on separe en phase d'entainement et phase test

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# on entraine les models

In [13]:
model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

# On Evalue les model sur une nouvelle set

In [14]:
y_pred = model_rf.predict(X_test)
accuracy_initial = accuracy_score(y_test, y_pred)

In [17]:
print(y_pred)

[1 0 0 ... 0 0 0]


In [23]:
print(f"Precision inital  du model: {accuracy_initial:.4f}")
print("\n Rapport de classification: \n", classification_report(y_test, y_pred))

Precision inital  du model: 0.7956

 Rapport de classification: 
               precision    recall  f1-score   support

           0       0.83      0.91      0.87      1036
           1       0.65      0.49      0.56       373

    accuracy                           0.80      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.78      0.80      0.79      1409



# Definisons les parametres du Grid

In [24]:
param_dist = {
    'n_estimators': np.arange(50, 200, 10),
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4]
}

# Initialiser RandomSearchCV 

In [25]:
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

## Entrainer sur les RandomSearchCV pour trouver la meilleur valeur 

In [26]:
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'max_depth': [None, 5, 10, 15],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10, 20],
                                        'n_estimators': array([ 50,  60,  70,  80,  90, 100, 110, 120, 130, 140, 150, 160, 170,
       180, 190])},
                   random_state=42, scoring='accuracy')

### Obtenir les meiilleurs parametres

In [29]:
best_params = random_search.best_params_
print(f"Meilleur Paramètres (RandomizedSearchCV): {best_params}")
print(f"Meilleur Score  (RandomizedSearchCV): {random_search.best_score_}")

Meilleur Paramètres (RandomizedSearchCV): {'n_estimators': 60, 'min_samples_split': 20, 'min_samples_leaf': 4, 'max_depth': 15}
Meilleur Score  (RandomizedSearchCV): 0.8015615420621873


# On affecte les meilleurs parametres

In [30]:
best_model = random_search.best_estimator_

## On predit sur les meilleurs model


In [31]:
y_pred_tuned = best_model.predict(X_test)
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)

In [33]:
print(f"Precision du model apres Tuning : {accuracy_tuned:.4f}")
print("\n Rapport de Classification (Tuned Model):\n", classification_report(y_test, y_pred_tuned))

Precision du model apres Tuning : 0.8098

 Rapport de Classification (Tuned Model):
               precision    recall  f1-score   support

           0       0.84      0.91      0.88      1036
           1       0.69      0.52      0.59       373

    accuracy                           0.81      1409
   macro avg       0.76      0.72      0.73      1409
weighted avg       0.80      0.81      0.80      1409



# Evaluer en utilisant la cross Validation

In [35]:
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='accuracy')

print(f"Score de precision de la Cross-Validation : {cv_scores}")
print(f"La Moyenne de la precision du Cross-Validation: {cv_scores.mean():.4f}")

Score de precision de la Cross-Validation : [0.80411639 0.80411639 0.7828247  0.80184659 0.80326705]
La Moyenne de la precision du Cross-Validation: 0.7992


# Nous avons appliqué des techniques d'optimisation des bout en bout , compris le pretraitement de l'engenering de features, nous avons utiliser la rzcherche aleatoir Ransdom search pour trouver les meilleurs hyper parametres , nous avons evalué en utilisant des validation croisé